This code validates the assertion in the paper that the total readout weight for each class is equal to O(sqrt(s)/(u-l))

In [ ]:
import torch

# Sparsity
s = 10
# Distribution of weights
p = 0.75
u = 0.1
l = -0.25
# Bias
b = 0.05

# Use a formula for bias that
# targets the mean of the distribution
b = -(s-1)* (p*u + (1-p)*l)







In [10]:
from collections import defaultdict
from math import comb
import scipy


def get_distribution(v0, v0_resp, v1, v1_resp, b_=None):
    if b_ is None:
        b_ = b
    r = defaultdict(float)
    remaining = s - v0 - v1
    for u_count in range(0, remaining + 1):
        l_count = remaining - u_count
        #prob = comb(remaining, u_count) * p**u_count * (1-p)**l_count
        prob = scipy.stats.binom.pmf(u_count, remaining, p)
        value = (
                    v0_resp * v0 +
                    v1_resp * v1 +
                    u * u_count +
                    l * l_count +
                    b_
        )
        r[value] += prob
    return r

def get_truth_table_for_class(v0_resp, v1_resp):
    table_keys = [(0,0), (0, 1), (1, 0), (1, 1)]

    result = {}
    for k in table_keys:
        dist = get_distribution(k[0], v0_resp, k[1], v1_resp)
        result[k] = sum(v * max(0, k) for k, v in dist.items())
    return result
            
def get_truth_tables():
    tt_A = get_truth_table_for_class(u, u)
    tt_B1 = get_truth_table_for_class(u, l)
    tt_B2 = get_truth_table_for_class(l, u)
    tt_C = get_truth_table_for_class(l, l)

    return torch.tensor([
        [tt_A[(0,0)], tt_B1[(0,0)], tt_B2[(0,0)], tt_C[(0,0)]],
        [tt_A[(0,1)], tt_B1[(0,1)], tt_B2[(0,1)], tt_C[(0,1)]],
        [tt_A[(1,0)], tt_B1[(1,0)], tt_B2[(1,0)], tt_C[(1,0)]],
        [tt_A[(1,1)], tt_B1[(1,1)], tt_B2[(1,1)], tt_C[(1,1)]],
    ], dtype=torch.double)



M = get_truth_tables()
target = torch.tensor([0, 0, 0, 1.0], dtype=torch.double)


weights = M.inverse() @ target

assert (M @ weights - target).norm() < 1e-6


print("Truth tables:", M)

print(f"A={weights[0]}, B1={weights[1]}, B2={weights[2]}, C={weights[3]}")




Truth tables: tensor([[2.0367, 2.0367, 2.0367, 2.0367],
        [2.4403, 0.8259, 2.4403, 0.8259],
        [2.4403, 2.4403, 0.8259, 0.8259],
        [2.9074, 1.0387, 1.0387, 0.1877]], dtype=torch.float64)
A=0.9824971703134014, B1=-0.9824971703134014, B2=-0.9824971703134014, C=0.9824971703134014


In [29]:
import plotly.express as px

def draw_line(fig, x, y_max, text):
    fig.add_shape(
        type="line",
        x0=x,
        y0=0,
        x1=x,
        y1=y_max,
        line=dict(
            color="LightSeaGreen",
            width=2,
        ),
    )
    fig.add_annotation(
        x=x,
        y=0,
        text=text,
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=16,
            color="#ffffff"
        ),
        align="center",
        bordercolor="#c7c7c7",
        borderwidth=2,
        borderpad=4,
        bgcolor="#ff7f0e",
        opacity=0.8,
        valign="top"
    )
def plot_distribution(v0_resp, v1_resp):
    r = get_distribution(1, v0_resp, 1, v1_resp, 0)
    x = [k for k,v in r.items()]
    y = [v for k,v in r.items()]

    fig = px.scatter(x=x, y=y)
    fig.update_traces( mode='lines+markers')
    draw_line(fig, -(b), max(y), "Class A")
    draw_line(fig, -(b+(u-l)), max(y), "Class B")
    draw_line(fig, -(b+(u-l)*2), max(y), "Class C")
    fig.update_layout(title_text=rf"$\text{{Value of }} X_i \text{{ for }} v_1 = v_2 = 1, s={s}$", xaxis_title="$X_i$", yaxis_title="Probability")
    fig.show()
    fig.write_html("plots/distribution.html", include_mathjax='cdn')


plot_distribution(u, u)

